# Data Exploration — chronologie, capteurs et visualiseur 2D

Les rubriques exploratoires de la version historique sont rétablies et adaptées
aux chargeurs contrôlés du projet. Les caractéristiques et scores des jalons 3
et 4 sont conservés ; le décodage du jalon 5 est ajouté à la fin. **Exécuter les cellules dans l'ordre avec Maj+Entrée.**

Ce notebook explore les observations, affiche les artefacts existants et permet
un diagnostic public explicite ; il n'entraîne ni ne règle de modèle. Tous les
tableaux, courbes et positions restent locaux : vider sorties, compteurs et
éventuelles pièces jointes avant publication.

Les cinq minutes annotées publiques servent au diagnostic de format, d'alignement
et de sens des labels. Aucun apprentissage, calibration ou choix de variante
ne doit en dépendre. Voir [le protocole](../docs/protocole.md).

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "config.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir Jupyter depuis le depot MYA ou analysis/")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src import config
from src.utils import load_atd, load_homography, load_smart_tagging, load_ground_truth
from src.utils.official import check_official_revision
from src.hygiene.masking import evaluated_mask
from src.features.team_assignment import from_tactical
from src.imputation.gap_analyzer import gap_features

RUN_PUBLIC_DIAGNOSTIC = False  # Activer uniquement le diagnostic, jamais un reglage.
VERIFY_BROADCAST = False       # Lecture facultative de la source TV independante.

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 10, "figure.dpi": 110})
GAME_NAME = "FC Barcelona vs Athletic Club"
PITCH_LENGTH, PITCH_WIDTH = config.PITCH_DIMS
FPS, TOTAL_FRAMES = config.FPS, config.TOTAL_FRAMES
print("Configuration :", config.MYA_DIR.name)
print("Revision officielle :", check_official_revision())
display(pd.DataFrame(config.check_data_assets().items(), columns=["fichier", "present"]))

## Chargement commun

Le tracking, le XML, les homographies et le Smart Tagging passent par les mêmes
interfaces que le programme. Aucun parseur de ligne ATD ou XML supplémentaire
n'est maintenu dans le notebook. Les identifiants vidéo restent ceux d'origine.

In [ ]:
atd = load_atd("tactical")
matrices, errors, valid, homography = load_homography("tactical")
smart = load_smart_tagging()
observations = from_tactical(atd, intervals=config.EVALUATED_INTERVALS,
    fps=FPS, pitch_dims=config.PITCH_DIMS)
eval_mask = evaluated_mask(TOTAL_FRAMES, config.EVALUATED_INTERVALS)
slot_teams = np.array([atd["team_mapping"][i] for i in range(atd["players"].shape[1])])
assert atd["players"].shape == (TOTAL_FRAMES, config.NUM_SLOTS, 3)
assert np.array_equal(atd["video_frames"], np.arange(TOTAL_FRAMES))
print("Joueurs :", atd["players"].shape, "Ballon :", atd["ball"].shape)
print("Unite de vitesse livree :", atd["speed_unit"],
      "Conflit documentaire :", atd["speed_documentation_conflict"])
print("Intervalles Smart Tagging :", smart.attrs["interval_convention"])

## 1. Vue d'ensemble et chronologie temporelle du match

Les bornes ci-dessous viennent des documents officiels, pas d'une annotation
humaine du match. Les intervalles sont **[début, fin[** en images vidéo.
Les deux plages évaluées peuvent contenir des arrêts de jeu ; évalué ne veut
pas dire ballon vivant. Les autres plages sont exclues de l'évaluation.

In [ ]:
periods = [
    ("Avant-match", *config.NON_EVALUATED_INTERVALS[0], False),
    ("Periode evaluee 1", *config.EVALUATED_INTERVALS[0], True),
    ("Mi-temps", *config.NON_EVALUATED_INTERVALS[1], False),
    ("Periode evaluee 2", *config.EVALUATED_INTERVALS[1], True),
    ("Apres-match", *config.NON_EVALUATED_INTERVALS[2], False),
]
rows = []
for name, start, end, evaluated in periods:
    usable_error = errors[start:end]
    rows.append({"plage": name, "debut_video": start, "fin_exclue": end,
        "duree_s": (end-start)/FPS, "evaluee": evaluated,
        "ballon_observe_pct": 100*atd["ball_observed"][start:end].mean(),
        "pistes_observees_moyenne": atd["players_observed"][start:end].sum(axis=1).mean(),
        "homographie_valide_pct": 100*valid[start:end].mean(),
        "erreur_nonnegative_mediane_px": float(np.nanmedian(usable_error)) if np.isfinite(usable_error).any() else np.nan})
period_summary = pd.DataFrame(rows)
display(period_summary)
kickoff_tags = smart[(smart["code"] == "SET PIECES") &
    smart["phase_type"].fillna("").str.contains("kick off", case=False)]
display(kickoff_tags[["code", "phase_type", "start_frame", "end_frame", "team_code"]])

In [ ]:
fig, ax = plt.subplots(figsize=(13, 2.5))
for name, start, end, evaluated in periods:
    ax.broken_barh([(start/FPS/60, (end-start)/FPS/60)], (0, 1),
        facecolors="tab:green" if evaluated else "lightgray", edgecolors="white")
    ax.text((start+end)/2/FPS/60, .5, name, ha="center", va="center", fontsize=8)
dead_start, dead_end = config.DEAD_ZONE_2HT_INTERVAL
ax.broken_barh([(dead_start/FPS/60, (dead_end-dead_start)/FPS/60)], (0, 1),
              facecolors="tab:orange", label="D publie avant reprise 2e periode")
ax.set(xlabel="Temps dans la video (min)", yticks=[], xlim=(0, TOTAL_FRAMES/FPS/60),
       title="Chronologie officielle — les plages vertes incluent du jeu vivant et mort")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.3))
plt.tight_layout(); plt.show()

In [ ]:
sample = np.arange(0, TOTAL_FRAMES, int(FPS))
time_min = sample/FPS/60
centre_distance = np.linalg.norm(atd["ball_metres"][sample, :2] -
    np.array(config.PITCH_DIMS)/2, axis=1)
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
axes[0].plot(time_min, atd["players_observed"][sample].sum(axis=1), lw=.8)
axes[0].set(ylabel="Pistes visibles", title="Verification des observations sur toute la video")
axes[1].plot(time_min, centre_distance, lw=.8, label="Distance ballon-centre (m)")
axes[1].set(ylabel="Distance (m)"); axes[1].legend()
axes[2].plot(time_min, errors[sample], lw=.8, label="Erreur exploitable (px)")
axes[2].set(xlabel="Temps video (min)", ylabel="Erreur (px)"); axes[2].legend()
for ax in axes:
    for _, start, end, evaluated in periods:
        if not evaluated: ax.axvspan(start/FPS/60, end/FPS/60, color="gray", alpha=.12)
plt.tight_layout(); plt.show()

## 2. Exploration du Smart Tagging (SportsCode XML)

Les événements sont automatiques : ils donnent un contexte, pas une vérité de
possession. La casse est normalisée, les équipes inconnues et les `N/A` restent
inconnus. La fenêtre d'une remise en jeu n'est pas un intervalle D garanti.

In [ ]:
print("Evenements :", len(smart), "Equipes inconnues :", smart["team_code"].isna().sum())
display(smart[["code", "start_frame", "end_frame", "duration_s", "team", "team_code", "phase_type"]].head())
set_pieces = smart[smart["code"] == "SET PIECES"]
display(set_pieces["phase_type"].fillna("inconnu").value_counts().rename_axis("type").to_frame("nombre"))
throw_ins = set_pieces[set_pieces["phase_type"].fillna("").str.contains("throw", case=False)]
display(throw_ins["team_code"].fillna("inconnue").value_counts().rename_axis("equipe_touche").to_frame("nombre"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
smart["code"].value_counts().sort_values().plot.barh(ax=axes[0], color="tab:blue")
axes[0].set(title="Frequence des codes automatiques", xlabel="Nombre d'evenements")
smart["team_code"].fillna("inconnue").value_counts().plot.bar(ax=axes[1], color="tab:orange", rot=0)
axes[1].set(title="Equipe indiquee, inconnus conserves", ylabel="Nombre d'evenements")
plt.tight_layout(); plt.show()

## 3. Exploration de l'homographie tactique

L'homographie transforme le terrain normalisé en pixels. L'ATD est déjà en
coordonnées terrain : il ne faut pas lui appliquer une homographie inverse.
Une matrice valide et une erreur de reprojection exploitable sont deux notions
distinctes. Les erreurs négatives livrées sont exclues des statistiques d'erreur.

In [ ]:
print("Matrices :", matrices.shape, "Valides :", int(valid.sum()))
print("Erreurs negatives livrees :", homography.attrs["negative_error_count"])
display(period_summary[["plage", "homographie_valide_pct", "erreur_nonnegative_mediane_px"]])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for period, (start, end) in enumerate(config.EVALUATED_INTERVALS, 1):
    values = errors[start:end]
    values = values[np.isfinite(values)]
    if len(values): axes[0].hist(values, bins=50, histtype="step", label=f"Periode {period}")
axes[0].set(xlabel="Erreur nonnegative (pixels)", ylabel="Images", title="Distribution de reprojection")
axes[0].legend()
axes[1].plot(sample/FPS/60, errors[sample], lw=.8)
axes[1].set(xlabel="Temps video (min)", ylabel="Erreur (pixels)", title="Dynamique de la qualite geometrique")
plt.tight_layout(); plt.show()

## 4. Exploration de l'ATD (Automatic Tracking Data)

Les 46 colonnes sont des pistes anonymes, pas 46 identités stables. Leur équipe
vient des références de colonnes XML. La disponibilité x/y est indépendante de
celle de la vitesse. Les positions sont exprimées en mètres par les chargeurs ;
la vitesse brute reste dans l'unité livrée, dont le désaccord documentaire est signalé.

In [ ]:
tracking_rows = []
for team in ("H", "A"):
    slots = slot_teams == team
    tracking_rows.append({"equipe": team, "pistes_du_XML": int(slots.sum()),
        "pistes_xy_moyennes_sur_plages_evaluees": atd["players_observed"][eval_mask][:, slots].sum(axis=1).mean(),
        "vitesses_livrees_disponibles_pct": 100*atd["players_speed_observed"][eval_mask][:, slots].mean()})
display(pd.DataFrame(tracking_rows))
print("Ballon absent sur images evaluees (%) :", 100*(~atd["ball_observed"][eval_mask]).mean())

In [ ]:
# Le meme analyseur que le jalon 3 : aucune concatenation des mi-temps.
gap_columns, last_seen, gap_runs = gap_features(atd["ball_observed"], observations.period_ids, FPS)
gap_table = pd.DataFrame(gap_runs, columns=["start_index", "end_index", "period_id", "length_frames", "left_censored", "right_censored"])
gap_table["duration_s"] = gap_table["length_frames"]/FPS
gap_table["classe_descriptive"] = pd.cut(gap_table["length_frames"],
    bins=[0, 15, 50, np.inf], labels=["court <=15", "moyen 16-50", "long >50"])
display(gap_table.groupby(["period_id", "classe_descriptive"], observed=False).agg(
    trous=("length_frames", "size"), images_absentes=("length_frames", "sum")))
if not gap_table.empty:
    display(gap_table.groupby("period_id").agg(
        duree_mediane_s=("duration_s", "median"), duree_max_s=("duration_s", "max"),
        censures_gauche=("left_censored", "sum"), censures_droite=("right_censored", "sum")))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if not gap_table.empty:
    longest = max(2, int(gap_table["length_frames"].max()))
    bins = np.geomspace(1, longest+1, 35)
    axes[0].hist(gap_table["length_frames"], bins=bins)
    axes[0].set_xscale("log")
axes[0].set(xlabel="Duree d'absence (images, echelle log)", ylabel="Trous",
            title="Absences du ballon, separees par periode")
for team in ("H", "A"):
    counts = atd["players_observed"][:, slot_teams == team].sum(axis=1).astype(float)
    counts[~eval_mask] = np.nan
    axes[1].plot(sample/FPS/60, counts[sample], lw=.8, label=team)
axes[1].set(xlabel="Temps video (min)", ylabel="Pistes observees", title="Support disponible par equipe")
axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Exploration de la vérité terrain publique (5 minutes, diagnostic facultatif)

Activer `RUN_PUBLIC_DIAGNOSTIC` dans la première cellule pour retrouver la
répartition, les durées et la frise des labels publics. Les intervalles sont
calculés automatiquement depuis les labels fournis. Aucun label nouveau ni
correction manuelle du match n'est créé. Ces observations ne règlent aucun modèle.

In [ ]:
gt_public = None
df_segments = None
if RUN_PUBLIC_DIAGNOSTIC:
    gt_public = load_ground_truth(public_window_only=True, purpose="diagnostic")
    assert len(gt_public) == config.PUBLIC_WINDOW_COUNT
    groups = (gt_public["label"].ne(gt_public["label"].shift()) |
              gt_public["frame"].diff().ne(1)).cumsum()
    df_segments = gt_public.groupby(groups, sort=False).agg(
        label=("label", "first"), start_frame=("frame", "first"),
        end_frame=("frame", "last"), length_frames=("frame", "size")).reset_index(drop=True)
    df_segments["end_frame"] += 1
    df_segments["duration_s"] = df_segments["length_frames"]/FPS
    possessions_only = df_segments[df_segments["label"].isin(["H", "A"])]
    display(gt_public["label"].value_counts().rename_axis("label").to_frame("images"))
    display(possessions_only[["label", "start_frame", "end_frame", "duration_s"]])
    display(possessions_only["duration_s"].describe().to_frame("duree_possessions_s"))
else:
    print("Diagnostic public desactive ; aucune annotation publique chargee.")

In [ ]:
if df_segments is not None:
    fig, ax = plt.subplots(figsize=(13, 2))
    colors = {"H": "tab:red", "A": "tab:blue", "D": "gray", "X": "lightgray"}
    first_frame = config.PUBLIC_DIAGNOSTIC_INTERVAL[0]
    used = set()
    for row in df_segments.itertuples():
        ax.broken_barh([((row.start_frame-first_frame)/FPS, row.duration_s)], (0, 1),
            facecolors=colors[row.label], label=row.label if row.label not in used else None)
        used.add(row.label)
    ax.set(xlabel="Secondes depuis le debut de la fenetre publique", yticks=[],
           xlim=(0, config.PUBLIC_WINDOW_COUNT/FPS), title="Labels publics fournis — diagnostic uniquement")
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.3), ncol=4)
    plt.tight_layout(); plt.show()

## 6. Visualiseur spatial 2D sur terrain de football

`plot_frame_situation(image_video, ax, titre)` affiche une image au choix depuis
les tableaux déjà validés. Les couleurs représentent les équipes du XML ; les
numéros éventuels sont des **indices de pistes**, jamais des dossards. Un ballon
absent reste absent et n'est pas replacé au centre. Les exemples utilisent
uniquement les bornes de coup d'envoi officiellement publiées, sans interprétation
manuelle de la possession ou du contact.

In [ ]:
def draw_pitch(ax=None, length=PITCH_LENGTH, width=PITCH_WIDTH):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6.5))
    ax.add_patch(Rectangle((0, 0), length, width, facecolor="#2e7d32", edgecolor="white", lw=2, zorder=1))
    ax.plot([length/2, length/2], [0, width], color="white", lw=2, zorder=2)
    ax.add_patch(Circle((length/2, width/2), 9.15, fill=False, edgecolor="white", lw=2, zorder=2))
    ax.plot(length/2, width/2, "o", color="white", markersize=4, zorder=2)
    for side in (0, 1):
        for depth, box_width in ((16.5, 40.32), (5.5, 18.32)):
            x = 0 if side == 0 else length-depth
            ax.add_patch(Rectangle((x, (width-box_width)/2), depth, box_width,
                                  fill=False, edgecolor="white", lw=1.5, zorder=2))
        ax.plot(11. if side == 0 else length-11., width/2, "o", color="white", markersize=4, zorder=2)
    # Conserver l'axe y du repere normalise livre, croissant vers le bas.
    ax.set(xlim=(-3, length+3), ylim=(width+3, -3), aspect="equal")
    ax.axis("off")
    return ax


def get_atd_frame(target_frame):
    if isinstance(target_frame, bool) or not isinstance(target_frame, (int, np.integer)) or not 0 <= target_frame < TOTAL_FRAMES:
        raise ValueError("Indice video entier hors limites")
    return {"players": atd["players_metres"][target_frame, :, :2].copy(),
            "ball": atd["ball_metres"][target_frame, :2].copy(),
            "observed": atd["players_observed"][target_frame].copy(),
            "teams": slot_teams.copy(), "frame": int(atd["video_frames"][target_frame])}


def plot_frame_situation(target_frame, ax=None, title_prefix="", show_slots=False):
    data = get_atd_frame(target_frame)
    ax = draw_pitch(ax)
    for team, color in (("H", "#e74c3c"), ("A", "#3498db")):
        selected = data["observed"] & (data["teams"] == team)
        points = data["players"][selected]
        ax.scatter(points[:, 0], points[:, 1], s=90, color=color, edgecolors="white", lw=1., label=team, zorder=4)
        if show_slots:
            for slot in np.flatnonzero(selected):
                ax.annotate(str(slot), data["players"][slot], color="white", fontsize=7, zorder=6)
    has_ball = np.isfinite(data["ball"]).all()
    if has_ball:
        ax.scatter(*data["ball"], s=90, color="#f1c40f", edgecolors="black", label="Ballon", zorder=5)
    ax.set_title(f"{title_prefix} — image video {target_frame} | ballon {'observe' if has_ball else 'indisponible'}", fontsize=10)
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.02), ncol=3)
    return ax

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_frame_situation(config.KICKOFF_1ST_HALF, axes[0], "Borne officielle 1re periode")
plot_frame_situation(config.KICKOFF_2ND_HALF, axes[1], "Borne officielle reprise 2e periode")
plt.tight_layout(); plt.show()
# Pour explorer une autre image, choisir son indice video puis appeler :
# plot_frame_situation(image_video, show_slots=True); plt.show()

## 7. Dynamique des métriques : références officielles en direct

Avec `RUN_PUBLIC_DIAGNOSTIC=True`, les quatre références déjà publiées sont
recalculées par **la commande de diagnostic partagée**. Aucun balayage de nouvelles
fenêtres ni sélection d'une variante n'est effectué. Les nombres de possessions
et durées sont lus dans les résultats officiels, pas inscrits comme constantes.

Le graphique match complet utilise exclusivement les agrégats publiés dans
`EVALUATION.md` à la révision épinglée. Ces labels privés ne sont pas disponibles.
La courbe illustre pourquoi les cinq minutes publiques ne sont pas un jeu de réglage.

In [ ]:
df_live_baselines = None
if RUN_PUBLIC_DIAGNOSTIC:
    from src.diagnostics import run as run_reference_diagnostic
    run_reference_diagnostic()
    board = pd.read_csv(config.CACHE_DIR / "jalon2/public_diagnostic/scoreboard.csv")
    baseline_names = ["baseline_raw", "baseline_smooth25", "baseline_smooth50", "baseline_smooth125"]
    df_live_baselines = board.set_index("submission").loc[baseline_names].reset_index()
    # Agregats publics d'EVALUATION.md ; aucune lecture du GT prive.
    published_full = {"baseline_raw": (.0486, .6532), "baseline_smooth25": (.1996, .6752),
                      "baseline_smooth50": (.2433, .6724), "baseline_smooth125": (.2242, .6356)}
    df_live_baselines["spellF1_match_complet_publie"] = df_live_baselines["submission"].map(lambda name: published_full[name][0])
    df_live_baselines["macroF1_match_complet_publie"] = df_live_baselines["submission"].map(lambda name: published_full[name][1])
    display(df_live_baselines[["submission", "spell_f1_mean", "macro_f1", "poss", "dur",
        "n_gt_spells", "spellF1_match_complet_publie", "macroF1_match_complet_publie"]])
else:
    print("Comparaison publique des references desactivee.")

In [ ]:
if df_live_baselines is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(df_live_baselines))
    axes[0].bar(x, pd.to_numeric(df_live_baselines["poss"]), color="tab:red")
    axes[0].axhline(int(df_live_baselines["n_gt_spells"].iloc[0]), ls="--", color="black", label="Reference publique H/A")
    axes[0].set_xticks(x, df_live_baselines["submission"], rotation=15)
    axes[0].set(ylabel="Possessions H/A", title="Fragmentation des references fournies")
    axes[0].legend()
    windows = [0, 25, 50, 125]
    axes[1].plot(windows, df_live_baselines["spell_f1_mean"], marker="o", label="Diagnostic public calcule")
    axes[1].plot(windows, df_live_baselines["spellF1_match_complet_publie"], marker="s", label="Agregat match complet publie")
    axes[1].axvline(50, color="green", ls="--", alpha=.6, label="Reference smooth50 fixee")
    axes[1].set(xlabel="Demi-largeur de fenetre (images)", ylabel="spellF1", title="Fenetre publique et agregats officiels")
    axes[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Afficher le diagnostic deja fige du jalon 4, sans reentrainement ni nouvel essai.
public_j4_path = config.CACHE_DIR / "observations/public_diagnostic/summary.json"
if RUN_PUBLIC_DIAGNOSTIC and public_j4_path.is_file():
    from src.refinement.exporter import sha256_file
    public_j4 = json.loads(public_j4_path.read_text(encoding="utf-8"))
    model_path = config.CACHE_DIR / "observations/model.json"
    if model_path.is_file() and sha256_file(model_path) == public_j4["model_sha256"]:
        display(pd.DataFrame(public_j4["metrics"]).T[["macro_f1", "spell_f1_mean", "n_pred_segments", "n_gt_segments"]])
    else:
        print("Diagnostic du jalon 4 historique : le modele courant ne correspond plus a son empreinte.")

## 8. Synthèse stratégique et plan d'action MYA

La chronologie contractuelle, la disponibilité des capteurs et les inconnus sont
mesurés ci-dessus. Les classes de durée des trous sont descriptives : elles ne
déterminent ni D, ni l'équipe, ni une règle automatique d'interpolation.

La proximité informe sur le contrôle potentiel, pas sur l'état vivant/mort.
Les pistes anonymes peuvent se fragmenter ; la qualité de projection est distincte
de la possession. Les variations de lissage des références publiées ne doivent
pas devenir une optimisation sur les cinq minutes publiques.

Les jalons 0 à 5 sont réalisés. Le modèle du jalon 4 améliore la classification
par image sur validation externe, mais ses segments restent trop fragmentés.
Le jalon 5 compare le décodage temporel sur les matchs externes autorisés ; voir son rapport de validation.
Les reconstructions, le pitch control et la fusion TV restent des essais séparés.

Voir [le plan](Implementation_Plan.md), [la validation J4](JALON_4_VALIDATION.md)
et [le diagnostic public figé](DIAGNOSTIC_PUBLIC_JALON_4.md). Aucun gain privé n'est présumé.

In [ ]:
exploration_summary = pd.DataFrame([
    {"controle": "Images video tactiques", "valeur": TOTAL_FRAMES},
    {"controle": "Images dans les plages evaluees", "valeur": int(eval_mask.sum())},
    {"controle": "Trous du ballon, periodes separees", "valeur": len(gap_table)},
    {"controle": "Tags avec equipe inconnue", "valeur": int(smart["team_code"].isna().sum())},
    {"controle": "Erreurs de reprojection negatives exclues", "valeur": homography.attrs["negative_error_count"]},
])
display(exploration_summary)

## 9. Vérification de la TV (facultatif)

La TV conserve sa chronologie, son XML et ses 51 pistes. Ce contrôle ne la
synchronise pas avec la tactique et ne suppose aucun décalage global.

In [ ]:
if VERIFY_BROADCAST:
    broadcast = load_atd("broadcast")
    assert broadcast["players"].shape == (config.BROADCAST_TOTAL_FRAMES, 51, 3)
    print("TV :", broadcast["players"].shape)
else:
    print("Verification TV facultative desactivee.")

## 10. Caractéristiques du jalon 3

Cette partie conserve l'API commune et les graphiques ajoutés au jalon 3. En
l'absence d'artefact, l'exploration précédente reste utilisable. Construire la
matrice avec `python -m src.build_features --source all` pour afficher cette partie.

In [ ]:
from src.build_features import load_matrix
features = None
feature_folder = config.CACHE_DIR / "features/tactical"
if (feature_folder / "manifest.json").is_file():
    features, feature_manifest, feature_schema = load_matrix(feature_folder)
    assert features["values"].shape == (TOTAL_FRAMES, 116)
    print("Images x caracteristiques :", features["values"].shape)
    dictionary = pd.DataFrame(feature_schema["features"])
    dictionary["images_valides"] = features["valid"].sum(axis=0)
    display(dictionary[["name", "unit", "group", "images_valides"]])
else:
    print("Artefact J3 absent : python -m src.build_features --source all")

In [ ]:
if features is not None:
    name = "collective_median_speed_mps"
    column = features["names"].tolist().index(name)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(features["frame_ids"][sample], features["values"][sample, column], lw=.8)
    ax.set(xlabel="Image video tactique", ylabel="Vitesse mediane (m/s)",
           title="Activite collective : les interruptions restent manquantes")
    plt.show()

## 11. Scores du jalon 4

Ces cellules relisent les scores vérifiés, sans entraîner de modèle ni ouvrir
de labels. Les probabilités sont non calibrées ; la confiance est leur
concentration, pas une mesure de justesse. Produire les scores avec les commandes
du jalon 4 décrites dans le README. Aucun décodage temporel n'est appliqué ici.

In [ ]:
from src.observation_pipeline import load_scores
scores = None
score_folder = config.CACHE_DIR / "observations/tactical/logistic"
if (score_folder / "manifest.json").is_file():
    scores, score_frames, score_periods, score_manifest = load_scores(score_folder)
    assert scores.probabilities.shape == (TOTAL_FRAMES, 3)
    assert np.array_equal(score_frames, np.arange(TOTAL_FRAMES))
    print("Scores images x classes :", scores.probabilities.shape)
    print("Ordre des classes :", score_manifest["class_order"])
    display(pd.DataFrame({"source_equipe": ["indisponible", "geometrie", "collectif", "maintien"],
        "images": [int(np.sum(scores.active & (scores.team_source == i))) for i in range(4)]}))
else:
    print("Artefact J4 absent : python -m src.observation_pipeline --tactical")

In [ ]:
if scores is not None:
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    for column, label in enumerate(["H", "A", "D"]):
        axes[0].plot(score_frames[sample], scores.probabilities[sample, column], label=label, lw=.8)
    axes[0].set(ylabel="Score", ylim=(0, 1), title="Scores d'observation H/A/D non calibres")
    axes[0].legend()
    axes[1].plot(score_frames[sample], scores.confidence[sample], lw=.8)
    axes[1].set(xlabel="Image video tactique", ylabel="Concentration", ylim=(0, 1))
    plt.tight_layout(); plt.show()

## 12. Décodage temporel du jalon 5

Cette section lit les labels déjà décodés, leurs marges et les profils de
frontières, sans ajuster de paramètres ni lire de labels publics. Construire
ces artefacts avec `python -m src.sequential_pipeline --fit --tactical` après
le jalon 4. Le CSV distinct est `submissions/MYA_jalon5.csv`.

Les marges sont des écarts d'objectif, pas des probabilités de justesse.
Les enveloppes de frontières ne sont pas des intervalles de confiance
statistiques. Les séquences et tableaux restent locaux et privés.

In [ ]:
from src.sequential_pipeline import load_sequence

sequential_results = {}
sequential_rows = []
for method in ("static", "dynamic"):
    folder = config.CACHE_DIR / "sequential" / "tactical" / method
    if not (folder / "manifest.json").is_file():
        print(f"Artefact J5 {method} absent : python -m src.sequential_pipeline --fit --tactical")
        continue
    sequence, sequence_manifest = load_sequence(folder)
    assert np.array_equal(sequence["frame_ids"], np.arange(TOTAL_FRAMES))
    sequential_results[method] = sequence
    changes = sum(int(np.count_nonzero(sequence["labels"][start+1:end] != sequence["labels"][start:end-1]))
                  for start, end in config.EVALUATED_INTERVALS)
    finite = sequence["margin"][np.isfinite(sequence["margin"])]
    sequential_rows.append({"methode": method, "images": len(sequence["labels"]),
        "changements_intra_periodes": changes,
        "marge_mediane_non_calibree": float(np.median(finite)) if len(finite) else np.nan})
if sequential_rows:
    display(pd.DataFrame(sequential_rows))

In [ ]:
if sequential_results:
    labels_to_plot = {}
    if scores is not None:
        labels_to_plot["Scores J4, maximum par image"] = scores.labels()
    labels_to_plot.update({f"Viterbi {name}": result["labels"] for name, result in sequential_results.items()})
    fig, axes = plt.subplots(len(labels_to_plot), 1, figsize=(13, 2*len(labels_to_plot)), sharex=True, squeeze=False)
    for ax, (name, labels) in zip(axes[:, 0], labels_to_plot.items()):
        values = pd.Series(labels).map({"H": 0., "A": 1., "D": 2.}).to_numpy()
        ax.step(np.arange(TOTAL_FRAMES)/FPS/60, values, where="post", lw=.7)
        ax.set(yticks=[0, 1, 2], yticklabels=["H", "A", "D"], ylim=(-.3, 2.3), title=name)
    axes[-1, 0].set_xlabel("Temps video (min) — coupures hors periodes")
    plt.tight_layout(); plt.show()